# Ensemble Methods

## Problem Definition

**Question.** Which classifier families must behave consistently before primary/meta comparison?

**Role in the workflow.** Serve as the shared Method Reference for logistic baseline, bagging, random forest, and boosting.

**Inputs.** Weighted development events and the split manifest.

**Outputs.** A configuration table only; model-specific scores remain in `primary_model.ipynb` and `meta_model.ipynb`.

**Why this method.** A common factory fixes preprocessing, balanced fitting, probability support, and random state 42 across tasks.

**Assumptions.** Ensembles can reduce variance or bias but do not remove dependence among overlapping financial events.

**Handoff.** The shared candidate factory to both model notebooks.


## Development Data and Common Behavior

The actual development schema and class balance are inspected without fitting a winner. This prevents duplicated methodological explanation and keeps all comparative evidence in the task-specific model notebooks.


In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.strategy_modeling.model_workflow import (
    build_candidate_classifiers,
    get_primary_feature_columns,
)

period = "2025-01-01_2025-12-31"
events = pd.read_parquet(PROJECT_ROOT / f"data/research_data/events/aapl_news_modeling_weighted_{period}.parquet")
manifest = pd.read_parquet(PROJECT_ROOT / "data/model_artifact/split_manifest.parquet")
development_starts = manifest.loc[manifest["partition"].eq("development"), "event_start"]
development = events[events["event_start"].isin(development_starts)].sort_values("event_start")
features = get_primary_feature_columns(development)

candidates = build_candidate_classifiers(random_state=42, n_jobs=1)
configuration = pd.DataFrame(
    [
        {
            "candidate": name,
            "estimator": pipeline.named_steps["model"].__class__.__name__,
            "scaled": "scaler" in pipeline.named_steps,
            "probability_output": hasattr(pipeline.named_steps["model"], "predict_proba"),
            "random_state": pipeline.named_steps["model"].get_params().get("random_state"),
        }
        for name, pipeline in candidates.items()
    ]
).set_index("candidate")

display(pd.Series({"development_events": len(development), "features": len(features), "negative_labels": int((development["direction_label"] == -1).sum()), "positive_labels": int((development["direction_label"] == 1).sum())}, name="value").to_frame())
display(configuration)


,value
development_events,978
features,53
negative_labels,481
positive_labels,497


,estimator,scaled,probability_output,random_state
candidate,,,,
logistic_regression,LogisticRegression,True,True,42
bagging,BaggingClassifier,False,True,42
random_forest,RandomForestClassifier,False,True,42
boosting,AdaBoostClassifier,False,True,42


## Results, Limitations, and Handoff

A fixed candidate set makes comparison fair but is not exhaustive. Logistic regression is the interpretability baseline; bagging and random forest target variance/redundancy; boosting targets sequential error correction.

The next notebook receives the four configured probability classifiers. No conclusion in this notebook is evidence of live-trading profitability.
